In [1]:
# ── CELL 1: ENVIRONMENT & EVALKIT-SRC VERIFICATION (INTERNET OFF) ──────
import os, sys, subprocess, json, hashlib, zipfile, shutil

RUN_MODE = os.environ.get("RUN_MODE", "full")
OUT_DIR = "Results/smoke" if RUN_MODE == "smoke" else "Results/final"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("Results/env", exist_ok=True)

EXPECTED_EVALKIT_SHA = "66d16ba721a0050820996c5998b3be9412057faa978fd98d331015f6065819ad"
print(f"Internet: OFF | RUN_MODE: {RUN_MODE} | Target: {OUT_DIR} | Expected evalkit SHA: {EXPECTED_EVALKIT_SHA}")

env_path = f"Results/env/final_01a_cv_gate_pass1_{RUN_MODE}_env.txt"
with open(env_path, 'w', encoding='utf-8') as f:
    f.write(f"Python Version: {sys.version}\n")
    try:
        gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'], text=True)
        f.write(f"GPU Info:\n{gpu_info}\n")
    except Exception as e:
        f.write(f"GPU Info: None ({e})\n")
print(f"Environment logged to: {env_path}")

# Auto-locate and extract evalkit-src archive
evalkit_ready = False
if os.path.exists("/kaggle/working/evalkit") and os.path.exists("/kaggle/working/configs"):
    print("[OK] evalkit already present in /kaggle/working.")
    evalkit_ready = True
elif os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in sorted(files):
            if "evalkit" in f.lower() and f.endswith(".zip"):
                zip_path = os.path.join(root, f)
                with open(zip_path, "rb") as zf:
                    found_sha = hashlib.sha256(zf.read()).hexdigest()
                print(f"[+] Found evalkit archive: {zip_path} (SHA-256: {found_sha})")
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall("/kaggle/working")
                evalkit_ready = True
                break
        if evalkit_ready:
            break
        if "evalkit" in dirs and "configs" in dirs:
            print(f"[+] Found unzipped evalkit at: {root}")
            if root != "/kaggle/working":
                if not os.path.exists("/kaggle/working/evalkit"):
                    shutil.copytree(os.path.join(root, "evalkit"), "/kaggle/working/evalkit")
                if not os.path.exists("/kaggle/working/configs"):
                    shutil.copytree(os.path.join(root, "configs"), "/kaggle/working/configs")
            evalkit_ready = True
            break

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

assert evalkit_ready or RUN_MODE == "smoke", "FATAL: evalkit-src not found in /kaggle/input!"
print("[OK] evalkit unpacked and loaded to sys.path.")

Internet: OFF | RUN_MODE: full | Target: Results/final | Expected evalkit SHA: 66d16ba721a0050820996c5998b3be9412057faa978fd98d331015f6065819ad
Environment logged to: Results/env/final_01a_cv_gate_pass1_full_env.txt
[+] Found unzipped evalkit at: /kaggle/input/datasets/omar10lfc/evalkit-src
[OK] evalkit unpacked and loaded to sys.path.


In [2]:
# ── CELL 2: PREFLIGHT INPUTS AUDIT & TIMING PROBE ────────────────────────
import os, glob, time, json, soundfile as sf

print("=== MOUNTED INPUT DIRECTORIES in /kaggle/input ===")
if os.path.exists("/kaggle/input"):
    for item in sorted(os.listdir("/kaggle/input")):
        print(f" - /kaggle/input/{item}")

# 1. Discover Common Voice directory (search for test.tsv)
COMMON_VOICE_DIR = None
for root, dirs, files in os.walk("/kaggle/input") if os.path.exists("/kaggle/input") else []:
    if "test.tsv" in files and ("clips" in dirs or "validated.tsv" in files or "train.tsv" in files):
        COMMON_VOICE_DIR = root
        break

if not COMMON_VOICE_DIR:
    for cand in [
        "/kaggle/input/common-voice-scripted-speech-25-0-arabic"
    ]:
        if os.path.exists(cand):
            COMMON_VOICE_DIR = cand
            break

assert (COMMON_VOICE_DIR is not None and os.path.exists(COMMON_VOICE_DIR)) or RUN_MODE == "smoke",     "FATAL: Missing Common Voice dataset in /kaggle/input"
print(f"[OK] Common Voice directory located: {COMMON_VOICE_DIR}")

# 2. Discover Merged Models directory (search for base/whisper-large-v3 or final-merged-models)
MERGED_MODELS_DIR = None
for root, dirs, files in os.walk("/kaggle/input") if os.path.exists("/kaggle/input") else []:
    if "base" in dirs and ("adapters" in dirs or "merged" in dirs or "ct2_int8" in dirs):
        MERGED_MODELS_DIR = root
        break
    if os.path.basename(root) == "final-merged-models":
        MERGED_MODELS_DIR = root
        break

if not MERGED_MODELS_DIR:
    for cand in [
        "/kaggle/input/final-merged-models",
        "/kaggle/input/zero-notebook/final-merged-models",
        "/kaggle/input/zero-notebook"
    ]:
        if os.path.exists(cand):
            MERGED_MODELS_DIR = cand
            break

assert (MERGED_MODELS_DIR is not None and os.path.exists(MERGED_MODELS_DIR)) or RUN_MODE == "smoke",     "FATAL: Missing final-merged-models directory. Mount 'final-merged-models' dataset or 'zero-notebook' output!"
print(f"[OK] Merged models directory located: {MERGED_MODELS_DIR}")

# Probe first 200 clips for timing estimate
print("\n=== AUDIO DURATION CALCULATION & 200-CLIP TIMING PROBE ===")
# Formula: Total Time = (N_clips * D_avg) / (Throughput * 3600)
# Pass 1 budget: <= 3.0 hours. Stops if projected total exceeds 3.5 hours.

=== MOUNTED INPUT DIRECTORIES in /kaggle/input ===
 - /kaggle/input/datasets
 - /kaggle/input/notebooks
[OK] Common Voice directory located: /kaggle/input/datasets/omar10lfc/common-voice-scripted-speech-25-0-arabic/15157464633 15157464443/cv-corpus-25.0-2026-03-09/ar
[OK] Merged models directory located: /kaggle/input/notebooks/omar10lfc/zero-notebook/final-merged-models

=== AUDIO DURATION CALCULATION & 200-CLIP TIMING PROBE ===


In [3]:
# ── CELL 3: PROTOCOL CONFIG & NORMALIZERS ──────────────────────────────
import json, os
from evalkit.normalize import normalize_arabic_legacy, normalize_arabic_v2
from evalkit.wer import compute_dual_wer_record, wer_breakdown
from evalkit.stats import save_result_file

cfg_path = "/kaggle/working/configs/final_protocol.json" if os.path.exists("/kaggle/working/configs/final_protocol.json") else "configs/final_protocol.json"
with open(cfg_path, 'r', encoding='utf-8') as f:
    PROTOCOL = json.load(f)
print(f"[OK] Protocol loaded: normalizer.primary={PROTOCOL['normalizer']['primary']}")

  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/csebuetnlp/xl-sum.git /tmp/pip-req-build-84j1d39e did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com/csebuetnlp/xl-sum.git /tmp/pip-req-build-84j1d39e did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


[OK] Protocol loaded: normalizer.primary=legacy


In [4]:
# ── CELL 4: GATE CHECK (TRAINING PROTOCOL REPLICA 400 CLIPS) ─────────────
import os, pandas as pd, numpy as np, torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import peft
from peft import PeftModel

# Fix incompatible torchao version check in PEFT
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao as peft_torchao
    peft_torchao.is_torchao_available = lambda: False
except Exception:
    pass

# Dynamically resolve paths using discovered roots
merged_root = globals().get('MERGED_MODELS_DIR') or "/kaggle/input/final-merged-models"
cv_root = globals().get('COMMON_VOICE_DIR') or "/kaggle/input/common-voice-scripted-speech-25-0-arabic"

base_path = os.path.join(merged_root, "base/whisper-large-v3")
adapter_path = os.path.join(merged_root, "adapters/whisper-large-v3-adapter")
tsv_path = os.path.join(cv_root, "test.tsv")

print(f"Base path    : {base_path} (exists: {os.path.exists(base_path)})")
print(f"Adapter path : {adapter_path} (exists: {os.path.exists(adapter_path)})")
print(f"TSV path     : {tsv_path} (exists: {os.path.exists(tsv_path)})")

if os.path.exists(tsv_path):
    df_raw = pd.read_csv(tsv_path, sep='\t', low_memory=False)
else:
    df_raw = pd.DataFrame({'path': [f'mock_{i}.mp3' for i in range(16)], 'sentence': ['هذا اختبار'] * 16})

df_raw = df_raw.dropna(subset=['sentence']).copy()
df_raw['clean_ref'] = df_raw['sentence'].apply(normalize_arabic_legacy)
df_valid = df_raw[df_raw['clean_ref'].str.len() > 2].reset_index(drop=True)
print(f"Common Voice Pool Size: {len(df_valid):,} (Expected: 10,506)")

subset_a = df_valid.sample(frac=1, random_state=42).iloc[:400].reset_index(drop=True)
first_5_ids = subset_a['path'].iloc[:5].tolist()
print(f"First 5 Clip IDs in Subset A: {first_5_ids}")

if RUN_MODE != "smoke":
    assert len(df_valid) == 10506, f"Pool size mismatch: got {len(df_valid)}, expected 10,506"

Base path    : /kaggle/input/notebooks/omar10lfc/zero-notebook/final-merged-models/base/whisper-large-v3 (exists: True)
Adapter path : /kaggle/input/notebooks/omar10lfc/zero-notebook/final-merged-models/adapters/whisper-large-v3-adapter (exists: True)
TSV path     : /kaggle/input/datasets/omar10lfc/common-voice-scripted-speech-25-0-arabic/15157464633 15157464443/cv-corpus-25.0-2026-03-09/ar/test.tsv (exists: True)
Common Voice Pool Size: 10,506 (Expected: 10,506)
First 5 Clip IDs in Subset A: ['common_voice_ar_23342586.mp3', 'common_voice_ar_24041068.mp3', 'common_voice_ar_27141031.mp3', 'common_voice_ar_19220379.mp3', 'common_voice_ar_19205134.mp3']


In [5]:
# ── CELL 5: GATE CHECK (SUBSET A - 400 CLIPS INFERENCE) ────────────────────
import os, time, torch, librosa
import pandas as pd
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel
from evalkit.normalize import normalize_arabic_legacy
from evalkit.wer import wer_breakdown

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# 1. Scan and Index Audio Clips (.mp3 / .wav)
audio_index = {}
t0_scan = time.time()
clips_dir = os.path.join(os.path.dirname(tsv_path), "clips")

if os.path.exists(clips_dir):
    print(f"Scanning clips directory: {clips_dir} ...")
    for fname in os.listdir(clips_dir):
        if fname.endswith(('.mp3', '.wav', '.ogg', '.m4a')):
            full_p = os.path.join(clips_dir, fname)
            stem = os.path.splitext(fname)[0]
            audio_index[stem] = full_p
            audio_index[fname] = full_p
    print(f"[OK] Indexed {len(audio_index)//2:,} audio clips from {clips_dir} in {time.time()-t0_scan:.1f}s.")
else:
    print("Scanning dataset directories recursively...")
    scan_roots = []
    if "COMMON_VOICE_DIR" in globals() and COMMON_VOICE_DIR:
        scan_roots.append(COMMON_VOICE_DIR)
    if "tsv_path" in globals() and tsv_path:
        scan_roots.append(os.path.dirname(tsv_path))
        scan_roots.append(os.path.dirname(os.path.dirname(tsv_path)))
        scan_roots.append(os.path.dirname(os.path.dirname(os.path.dirname(tsv_path))))
    scan_roots.extend([
        "/kaggle/input/common-voice-scripted-speech-25-0-arabic",
        "/kaggle/input"
    ])
    seen_roots = set()
    for r in scan_roots:
        norm = os.path.normpath(r) if r else ""
        if norm and norm not in seen_roots and os.path.exists(norm):
            seen_roots.add(norm)
            for root, dirs, files in os.walk(norm):
                for fname in files:
                    if fname.endswith(('.mp3', '.wav', '.ogg', '.m4a')):
                        full_p = os.path.join(root, fname)
                        stem = os.path.splitext(fname)[0]
                        audio_index[stem] = full_p
                        audio_index[fname] = full_p
    print(f"[OK] Indexed {len(audio_index)//2:,} audio clips in {time.time()-t0_scan:.1f}s.")

def resolve_audio_path(raw):
    s = str(raw)
    stem = os.path.splitext(os.path.basename(s))[0]
    fname = os.path.basename(s)
    if clips_dir and os.path.exists(clips_dir):
        p = os.path.join(clips_dir, fname)
        if os.path.exists(p):
            return p
    return audio_index.get(stem) or audio_index.get(fname)

# Validate first clip of Subset A
first_clip_id = subset_a['path'].iloc[0]
first_clip_path = resolve_audio_path(first_clip_id)
print(f"Sample Clip ID : {first_clip_id}")
print(f"Resolved Path  : {first_clip_path}")
print(f"Exists on disk : {os.path.exists(first_clip_path) if first_clip_path else False}")

if RUN_MODE != "smoke":
    assert first_clip_path is not None and os.path.exists(first_clip_path), \
        f"FATAL: Could not locate audio file for clip {first_clip_id}!"

# 2. Load Whisper-Large-v3 (Pre-Merged or Base + LoRA)
merged_path = os.path.join(merged_root, "merged/whisper-large-v3")
processor = None
model = None

if os.path.exists(merged_path):
    try:
        print(f"Loading pre-merged Whisper-Large-v3 from: {merged_path}...")
        processor = WhisperProcessor.from_pretrained(merged_path)
        model = WhisperForConditionalGeneration.from_pretrained(
            merged_path, torch_dtype=dtype
        ).to(device)
        print(f"[OK] Successfully loaded pre-merged model.")
    except Exception as e:
        print(f"[NOTICE] Direct load failed ({e}), falling back to base + adapter.")
        model = None

if model is None:
    print(f"Loading base model from: {base_path}...")
    processor = WhisperProcessor.from_pretrained(base_path)
    base_model = WhisperForConditionalGeneration.from_pretrained(
        base_path, torch_dtype=dtype
    ).to(device)
    print(f"Attaching and fusing LoRA adapter from: {adapter_path}...")
    model = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
    print(f"[OK] Successfully fused LoRA adapter into base model.")

model.eval()
model.generation_config.language = "ar"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = None
model.generation_config.begin_suppress_tokens = None

# 3. Audio Transcription Helper
def transcribe_audio_clip(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    speech, sr = librosa.load(audio_path, sr=16000, mono=True)
    duration = len(speech) / 16000.0
    inputs = processor(
        speech[:480000], sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device, dtype=dtype)
    with torch.no_grad():
        gen_ids = model.generate(
            inputs, language="ar", task="transcribe", num_beams=1, do_sample=False
        )
    text = processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    return text, duration

# 4. Transcribe Subset A (Gate Check)
eval_subset = subset_a.iloc[:8] if RUN_MODE == "smoke" else subset_a
n_gate = len(eval_subset)
print(f"\n--- Transcribing Gate Subset A ({n_gate} clips) on {device} ---")

gate_s, gate_d, gate_i, gate_h, gate_n = 0, 0, 0, 0, 0
gate_audio_s = 0.0
t0_gate = time.time()

for idx, (_, row) in enumerate(eval_subset.iterrows()):
    clip_path = resolve_audio_path(row['path'])
    hyp_raw, dur = transcribe_audio_clip(clip_path)
    gate_audio_s += dur
    
    ref_norm = normalize_arabic_legacy(row['sentence'])
    hyp_norm = normalize_arabic_legacy(hyp_raw)
    s, d, ins, h, n, _ = wer_breakdown(ref_norm, hyp_norm)
    
    gate_s += s
    gate_d += d
    gate_i += ins
    gate_h += h
    gate_n += n
    
    if (idx + 1) % 50 == 0 or (idx + 1) == n_gate:
        curr_wer = ((gate_s + gate_d + gate_i) / gate_n * 100.0) if gate_n > 0 else 0.0
        print(f"  [Gate] Processed {idx+1}/{n_gate} clips | Running WER: {curr_wer:.2f}% | Elapsed: {time.time()-t0_gate:.1f}s")

elapsed_gate = time.time() - t0_gate
gate_wer = ((gate_s + gate_d + gate_i) / gate_n * 100.0) if gate_n > 0 else 0.0
gate_throughput = gate_audio_s / elapsed_gate if elapsed_gate > 0 else 0.0

print(f"\nGate Measured WER: {gate_wer:.2f}% (Throughput: {gate_throughput:.2f}x real-time)")
diff = abs(gate_wer - 12.51)
print(f"Delta vs 12.51% baseline: {diff:.2f} pp")

tolerance = 5.0 if RUN_MODE == "smoke" else 1.0
assert diff <= tolerance, f"GATE STOPPED: Measured WER ({gate_wer:.2f}%) differs from training 12.51% by {diff:.2f} pp (> {tolerance:.2f} pp)"
print(f"[GATE PASSED] Replicated 12.51% within tolerance (diff: {diff:.2f} pp). Proceeding to Pass 1.")


Scanning dataset directories recursively...
[OK] Indexed 136,140 audio clips in 11.7s.
Sample Clip ID : common_voice_ar_23342586.mp3
Resolved Path  : /kaggle/input/datasets/omar10lfc/common-voice-scripted-speech-25-0-arabic/15157464351 15157464155/cv-corpus-25.0-2026-03-09/ar/clips/common_voice_ar_23342586.mp3
Exists on disk : True
Loading pre-merged Whisper-Large-v3 from: /kaggle/input/notebooks/omar10lfc/zero-notebook/final-merged-models/merged/whisper-large-v3...


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

[OK] Successfully loaded pre-merged model.

--- Transcribing Gate Subset A (400 clips) on cuda ---


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  [Gate] Processed 50/400 clips | Running WER: 9.77% | Elapsed: 50.1s
  [Gate] Processed 100/400 clips | Running WER: 11.88% | Elapsed: 81.8s
  [Gate] Processed 150/400 clips | Running WER: 12.86% | Elapsed: 115.3s
  [Gate] Processed 200/400 clips | Running WER: 12.82% | Elapsed: 146.3s
  [Gate] Processed 250/400 clips | Running WER: 12.70% | Elapsed: 182.6s
  [Gate] Processed 300/400 clips | Running WER: 12.76% | Elapsed: 216.1s
  [Gate] Processed 350/400 clips | Running WER: 13.11% | Elapsed: 249.2s
  [Gate] Processed 400/400 clips | Running WER: 13.03% | Elapsed: 285.3s

Gate Measured WER: 13.03% (Throughput: 6.29x real-time)
Delta vs 12.51% baseline: 0.52 pp
[GATE PASSED] Replicated 12.51% within tolerance (diff: 0.52 pp). Proceeding to Pass 1.


In [6]:
# ── CELL 6: PASS 1 FULL POOL EVALUATION (PYTORCH FP16 GREEDY) ─────────────
import math
from evalkit.normalize import normalize_arabic_legacy, normalize_arabic_v2
from evalkit.wer import compute_dual_wer_record
from evalkit.stats import bootstrap_ci, save_result_file

pass1_out_file = os.path.join(OUT_DIR, "cv_pass1_pytorch_fp16.json")
ckpt_csv_file = os.path.join(OUT_DIR, "cv_pass1_checkpoint.csv")
print(f"Pass 1 target output: {pass1_out_file}")
print(f"Pass 1 checkpoint CSV: {ckpt_csv_file}")

active_pool = df_valid.iloc[:8].copy() if RUN_MODE == "smoke" else df_valid.copy()
total_pool_clips = len(active_pool)

# 1. Resume from Checkpoint if Available
processed_paths = set()
checkpoint_rows = []
if os.path.exists(ckpt_csv_file):
    try:
        df_prev = pd.read_csv(ckpt_csv_file)
        processed_paths = set(df_prev['path'].astype(str))
        checkpoint_rows = df_prev.to_dict(orient='records')
        print(f"[RESUME] Loaded {len(processed_paths):,} already-transcribed clips from {ckpt_csv_file}")
    except Exception as e:
        print(f"[NOTICE] Could not read existing checkpoint ({e}); starting fresh.")
        checkpoint_rows = []

# 2. Main Transcription Loop
checkpoint_interval = 4 if RUN_MODE == "smoke" else 500
t0_pass1 = time.time()
new_rows = []

print(f"\nStarting Pass 1 inference across {total_pool_clips:,} clips (remaining: {total_pool_clips - len(processed_paths):,})...")

for idx, (_, row) in enumerate(active_pool.iterrows()):
    p_str = str(row['path'])
    if p_str in processed_paths:
        continue
    
    clip_path = resolve_audio_path(row['path'])
    t_clip_start = time.time()
    hyp_raw, dur = transcribe_audio_clip(clip_path)
    compute_dur = time.time() - t_clip_start
    
    # Compute dual metrics (legacy & v2)
    dual = compute_dual_wer_record(row['sentence'], hyp_raw)
    
    record = {
        'path': p_str,
        'audio_duration_s': round(dur, 2),
        'compute_duration_s': round(compute_dur, 4),
        'raw_ref': row['sentence'],
        'raw_hyp': hyp_raw,
        'legacy_s': dual['legacy_s'],
        'legacy_d': dual['legacy_d'],
        'legacy_i': dual['legacy_i'],
        'legacy_h': dual['legacy_h'],
        'legacy_n': dual['legacy_n'],
        'legacy_wer': dual['legacy_wer'],
        'v2_s': dual['v2_s'],
        'v2_d': dual['v2_d'],
        'v2_i': dual['v2_i'],
        'v2_h': dual['v2_h'],
        'v2_n': dual['v2_n'],
        'v2_wer': dual['v2_wer'],
    }
    new_rows.append(record)
    checkpoint_rows.append(record)
    processed_paths.add(p_str)
    
    # Checkpoint writing
    if len(new_rows) >= checkpoint_interval or len(checkpoint_rows) == total_pool_clips:
        df_chunk = pd.DataFrame(new_rows)
        write_header = not os.path.exists(ckpt_csv_file)
        df_chunk.to_csv(ckpt_csv_file, mode='a', header=write_header, index=False, encoding='utf-8')
        new_rows = []
        
        # Progress stats
        tot_aud = sum(r['audio_duration_s'] for r in checkpoint_rows)
        tot_cmp = sum(r['compute_duration_s'] for r in checkpoint_rows)
        rt_factor = tot_aud / tot_cmp if tot_cmp > 0 else 0.0
        tot_s_leg = sum(r['legacy_s'] for r in checkpoint_rows)
        tot_d_leg = sum(r['legacy_d'] for r in checkpoint_rows)
        tot_i_leg = sum(r['legacy_i'] for r in checkpoint_rows)
        tot_n_leg = sum(r['legacy_n'] for r in checkpoint_rows)
        running_wer = ((tot_s_leg + tot_d_leg + tot_i_leg) / tot_n_leg * 100.0) if tot_n_leg > 0 else 0.0
        
        print(f"  [Pass 1] Processed {len(checkpoint_rows):,}/{total_pool_clips:,} clips | Running WER (legacy): {running_wer:.2f}% | Throughput: {rt_factor:.2f}x")

# Flush any remaining new rows
if new_rows:
    df_chunk = pd.DataFrame(new_rows)
    write_header = not os.path.exists(ckpt_csv_file)
    df_chunk.to_csv(ckpt_csv_file, mode='a', header=write_header, index=False, encoding='utf-8')
    new_rows = []

# 3. Calculate Final Aggregate Statistics
total_audio_s = sum(r['audio_duration_s'] for r in checkpoint_rows)
total_compute_s = sum(r['compute_duration_s'] for r in checkpoint_rows)
throughput = total_audio_s / total_compute_s if total_compute_s > 0 else 0.0
rtf = total_compute_s / total_audio_s if total_audio_s > 0 else 0.0

tot_s_leg = sum(r['legacy_s'] for r in checkpoint_rows)
tot_d_leg = sum(r['legacy_d'] for r in checkpoint_rows)
tot_i_leg = sum(r['legacy_i'] for r in checkpoint_rows)
tot_h_leg = sum(r['legacy_h'] for r in checkpoint_rows)
tot_n_leg = sum(r['legacy_n'] for r in checkpoint_rows)
overall_wer_leg = ((tot_s_leg + tot_d_leg + tot_i_leg) / tot_n_leg * 100.0) if tot_n_leg > 0 else 0.0

tot_s_v2 = sum(r['v2_s'] for r in checkpoint_rows)
tot_d_v2 = sum(r['v2_d'] for r in checkpoint_rows)
tot_i_v2 = sum(r['v2_i'] for r in checkpoint_rows)
tot_h_v2 = sum(r['v2_h'] for r in checkpoint_rows)
tot_n_v2 = sum(r['v2_n'] for r in checkpoint_rows)
overall_wer_v2 = ((tot_s_v2 + tot_d_v2 + tot_i_v2) / tot_n_v2 * 100.0) if tot_n_v2 > 0 else 0.0

# 4. Bootstrap Confidence Intervals on Clip-Level WERs
clip_wers_leg = [r['legacy_wer'] * 100.0 for r in checkpoint_rows]
clip_wers_v2 = [r['v2_wer'] * 100.0 for r in checkpoint_rows]
ci_leg = bootstrap_ci(clip_wers_leg, n_bootstrap=1000)
ci_v2 = bootstrap_ci(clip_wers_v2, n_bootstrap=1000)

pass1_results = {
    "pass": "Pass 1: Common Voice Full Pool (PyTorch FP16 Greedy)",
    "run_mode": RUN_MODE,
    "model": "Whisper-Large-v3 (Fine-Tuned Arabic)",
    "precision": "PyTorch FP16",
    "decoder": "Greedy (beam_size=1, temperature=0.0)",
    "num_clips": len(checkpoint_rows),
    "total_audio_seconds": round(total_audio_s, 2),
    "total_audio_hours": round(total_audio_s / 3600.0, 3),
    "total_compute_seconds": round(total_compute_s, 2),
    "total_compute_hours": round(total_compute_s / 3600.0, 3),
    "throughput_realtime": round(throughput, 2),
    "rtf": round(rtf, 4),
    "gate_subset_a": {
        "gate_wer_measured": round(gate_wer, 2),
        "baseline_wer": 12.51,
        "delta_pp": round(abs(gate_wer - 12.51), 2),
        "status": "PASSED"
    },
    "legacy_normalizer": {
        "corpus_wer": round(overall_wer_leg, 2),
        "substitutions": tot_s_leg,
        "deletions": tot_d_leg,
        "insertions": tot_i_leg,
        "hits": tot_h_leg,
        "total_words": tot_n_leg,
        "clip_level_bootstrap_ci_95": ci_leg
    },
    "v2_normalizer": {
        "corpus_wer": round(overall_wer_v2, 2),
        "substitutions": tot_s_v2,
        "deletions": tot_d_v2,
        "insertions": tot_i_v2,
        "hits": tot_h_v2,
        "total_words": tot_n_v2,
        "clip_level_bootstrap_ci_95": ci_v2
    }
}

# 5. Save Verified Result File
save_result_file(pass1_out_file, pass1_results, config=PROTOCOL, device=str(device))

print("\n" + "="*70)
print(f"[OK] Pass 1 completed successfully!")
print(f"Results written to: {pass1_out_file}")
print(f"Checkpoint written to: {ckpt_csv_file}")
print(f"Total Clips Transcribed: {len(checkpoint_rows):,}")
print(f"Total Audio Duration: {total_audio_s/3600.0:.2f} hours")
print(f"Corpus WER (Legacy): {overall_wer_leg:.2f}%")
print(f"Corpus WER (V2): {overall_wer_v2:.2f}%")
print(f"Throughput: {throughput:.2f}x real-time (RTF: {rtf:.4f})")
print("="*70)

Pass 1 target output: Results/final/cv_pass1_pytorch_fp16.json
Pass 1 checkpoint CSV: Results/final/cv_pass1_checkpoint.csv

Starting Pass 1 inference across 10,506 clips (remaining: 10,506)...
  [Pass 1] Processed 500/10,506 clips | Running WER (legacy): 19.33% | Throughput: 7.18x
  [Pass 1] Processed 1,000/10,506 clips | Running WER (legacy): 17.71% | Throughput: 6.90x
  [Pass 1] Processed 1,500/10,506 clips | Running WER (legacy): 16.84% | Throughput: 6.93x
  [Pass 1] Processed 2,000/10,506 clips | Running WER (legacy): 16.70% | Throughput: 6.92x
  [Pass 1] Processed 2,500/10,506 clips | Running WER (legacy): 16.35% | Throughput: 6.93x
  [Pass 1] Processed 3,000/10,506 clips | Running WER (legacy): 16.51% | Throughput: 6.89x
  [Pass 1] Processed 3,500/10,506 clips | Running WER (legacy): 16.43% | Throughput: 6.88x
  [Pass 1] Processed 4,000/10,506 clips | Running WER (legacy): 16.01% | Throughput: 6.86x
  [Pass 1] Processed 4,500/10,506 clips | Running WER (legacy): 15.56% | Through